In [1]:
import os
from pathlib import Path

First we need to define where PLM_interact finds our input data and where the output files are written to. You can see these files in the file browser on the left. If you change these names, remember to change them in the analysis notebook as well.


In [ ]:
WORKING_DIR = Path.cwd() / "PLM_interact"  # created by the user
OUTPUT_DIR = WORKING_DIR / "output"  # created by the notebook

Next we need to upload our sequence pairs. A description of the `.csv` format can be found [here](https://github.com/liudan111/PLM-interact/tree/main#1-ppi-inference-with-multi-gpus). You also need to tell PLM-interact the length of the longest sequence.

In [ ]:
list_of_sequence_pairs = WORKING_DIR / "test.csv"
max_length = 1520

Now we can prepare a bash file that calls PLM-interact.

In [ ]:
RUN_FILE = WORKING_DIR / "run.sh"

run_file = f"""
#!/bin/bash

module load devel/miniforge/24.9.2
conda activate /mnt/sds-hd/sd25g005/PLMinteract

! torchrun --nproc_per_node=1 -m PLMinteract inference_PPI \\
--seed 2 \\
--batch_size_val 1 \\
--test_filepath {list_of_sequence_pairs} \\
--resume_from_checkpoint /mnt/sds-hd/sd25g005/PLMinteract/download_huggingface_folder/PLM-interact-650M-humanV11/pytorch_model.bin \\
--output_filepath {OUTPUT_DIR}/ \\
--offline_model_path /mnt/sds-hd/sd25g005/PLMinteract/download_huggingface_folder/offline/ \\
--model_name esm2_t33_650M_UR50D \\
--embedding_size 1280 --max_length {max_length}
"""

with open(RUN_FILE, "w") as file:
    file.write(run_file)

Execute the cell below to start the PLM-interact:

In [5]:
os.system(f'echo "Running file {RUN_FILE}"')
os.system(f"bash {RUN_FILE}")

Running file run.sh


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


256

If you want to have your input sequences and the output scores in one `.csv` file, execute the following cells:

In [ ]:
# ! pip install pandas
import pandas as pd

input_csv = pd.read_csv(list_of_sequence_pairs)
output_csv = pd.read_csv(OUTPUT_DIR / "pred_scores.csv", header=None)

results_csv = pd.merge(input_csv, output_csv, left_index=True, right_index=True)
results_csv.to_csv(OUTPUT_DIR / "pred_scores_w_sequences.csv", index=False)